# Do the invariance decision rules track *practical* significance?

The conventional ΔCFI/ΔRMSEA cutoffs were proposed as practical-significance screens
(Cheung & Rensvold, 2002), not as hypothesis tests. This notebook therefore compares every
decision rule in the project — conventional cutoffs, the scaled-Δχ² LRT, RMSEA_D,
McNeish-style dynamic cutoffs (`NB_calibrated_criteria.ipynb`) and the permutation ladder
itself — against the *score-level consequences* of non-invariance rather than against each
other.

Consequences are quantified for **all 48 scale × pair cells** (not only the permutation
failures, as in `NB_invariance_effect_sizes.ipynb`) with the metric-model DIF decomposition
(`hitop_cfa.effect_size.dif_effect_sizes`): net and gross DIF-induced sum-score bias as a
percentage of scale range, whether the net-bias 95% CI excludes zero, and the largest item
dMACS. For configural-failing scales the ωH-based criterion-correlation attenuation from the
dimensionality analysis is joined as a secondary consequence measure.

Limitation carried into the write-up: the DIF-bias measures capture threshold/intercept DIF
acting on sum scores. Loading non-invariance biases correlations rather than sums and is
not represented, so metric-level flags are inconsequential *by construction* on these
measures; configural failures are represented only through the attenuation measure.

Outputs in `data/practical_significance/`.

In [1]:
import os
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")
import datetime, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

import hitop_cfa
from hitop_cfa.r_env import ro, set_seeds, silence_r
from hitop_cfa.effect_size import dif_effect_sizes
from hitop_cfa.scales import OTHER_SCALES, ORIG_ITEMS

dat_dir = Path('../data'); fin_dir = dat_dir / 'finaldata'
out_dir = dat_dir / 'practical_significance'; out_dir.mkdir(exist_ok=True)
temp_dir = out_dir / 'temp'; temp_dir.mkdir(exist_ok=True)
progress_log = out_dir / 'progress.log'
def log(msg):
    line = f"[{datetime.datetime.now().isoformat(timespec='seconds')}] {msg}"
    print(line, flush=True)
    with open(progress_log, 'a') as fh: fh.write(line + '\n')
set_seeds(12345); silence_r()
LEVELS = ['configural', 'thresholds', 'metric', 'scalar', 'strict']

In [2]:
data_val = pd.read_csv(fin_dir / 'dat_val.csv')
data_gp = pd.read_csv(fin_dir / 'dat_gp_grid1st_full.csv')
data_en = pd.read_csv(fin_dir / 'dat_en_grid1st_full.csv')
datasets = {'val_gp': pd.concat([data_val, data_gp]),
            'val_en': pd.concat([data_val, data_en]),
            'gp_en': pd.concat([data_gp, data_en])}
pair_groups = {'val_gp': ('validation', 'genpop'), 'val_en': ('validation', 'enriched'), 'gp_en': ('genpop', 'enriched')}  # labels as in dimensionality_effect_sizes.csv
ALL_SCALE_ITEMS = list(OTHER_SCALES.items()) + list(ORIG_ITEMS.items())

## 1. DIF effect sizes for every cell

In [3]:
rows, item_rows = [], []
for pair, data in datasets.items():
    scale_items = ALL_SCALE_ITEMS if pair == 'gp_en' else list(ORIG_ITEMS.items())
    for scale, formula in scale_items:
        items = formula.split('=~', 1)[1].strip().split(' + ')
        t0 = time.time()
        try:
            per_item, summ = dif_effect_sizes(scale, items, data, temp_dir / 'es_temp.csv', n_draws=1000)
        except Exception as exc:
            log(f'{pair} {scale}: ERROR {exc!r}'); rows.append(dict(pair=pair, scale=scale, converged=False, error=repr(exc))); continue
        if per_item is None or not summ.get('converged', False):
            log(f'{pair} {scale}: metric model did not converge'); rows.append(dict(pair=pair, scale=scale, converged=False)); continue
        smax = summ['scale_max_points']
        rows.append(dict(
            pair=pair, scale=scale, converged=True, n_items=len(items), scale_max=smax,
            alpha=summ['alpha'], real_gap_pts=summ['real_gap_points'],
            net_bias_pts=summ['net_sum_bias_points'], net_lo=summ['net_ci'][0], net_hi=summ['net_ci'][1],
            gross_bias_pts=summ['gross_abs_bias_points'],
            net_bias_pct=100 * summ['net_sum_bias_points'] / smax,
            abs_net_bias_pct=100 * abs(summ['net_sum_bias_points']) / smax,
            gross_bias_pct=100 * summ['gross_abs_bias_points'] / smax,
            net_ci_excl0=bool(summ['net_ci'][0] > 0 or summ['net_ci'][1] < 0),
            max_dmacs=float(per_item['dMACS'].max()),
            max_dmacs_item=per_item.loc[per_item['dMACS'].idxmax(), 'item'],
            n_items_dif_ci_excl0=int(((per_item.signed_lo > 0) | (per_item.signed_hi < 0)).sum()),
            seconds=round(time.time() - t0, 1)))
        pi = per_item.copy(); pi.insert(0, 'scale', scale); pi.insert(0, 'pair', pair); item_rows.append(pi)
        log(f"{pair} {scale}: gross={rows[-1]['gross_bias_pct']:.1f}% net={rows[-1]['net_bias_pct']:+.1f}% maxdMACS={rows[-1]['max_dmacs']:.2f} [{rows[-1]['seconds']}s]")
        pd.DataFrame(rows).to_csv(out_dir / 'dif_all_cells.csv', index=False)
es = pd.DataFrame(rows)
pd.concat(item_rows).to_csv(out_dir / 'dif_all_items.csv', index=False)
es.to_csv(out_dir / 'dif_all_cells.csv', index=False)
log(f'DIF effect sizes done for {len(es)} cells; converged {int(es.converged.sum())}')

[2026-10-02T10:23:05] val_gp anhedonic_depression: gross=2.2% net=-0.3% maxdMACS=0.23 [1.5s]


[2026-10-02T10:23:06] val_gp anxious_worry: gross=3.8% net=-0.7% maxdMACS=0.26 [1.1s]


[2026-10-02T10:23:07] val_gp appetite_gain: gross=1.2% net=-0.1% maxdMACS=0.08 [0.8s]


[2026-10-02T10:23:07] val_gp appetite_loss: gross=2.7% net=+0.7% maxdMACS=0.17 [0.6s]


[2026-10-02T10:23:08] val_gp cognitive_problems: gross=2.2% net=-0.4% maxdMACS=0.17 [0.7s]


[2026-10-02T10:23:09] val_gp hyposomnia: gross=2.0% net=-0.8% maxdMACS=0.16 [0.9s]


... (39 outputs left out)

[2026-10-02T10:23:44] gp_en situational_phobia: gross=5.1% net=-0.1% maxdMACS=0.29 [0.7s]


[2026-10-02T10:23:46] gp_en social_anxiety: gross=3.1% net=+0.7% maxdMACS=0.17 [1.2s]


[2026-10-02T10:23:47] gp_en well_being: gross=2.3% net=+0.4% maxdMACS=0.18 [1.2s]


[2026-10-02T10:23:47] DIF effect sizes done for 48 cells; converged 48


## 2. Join the dimensionality attenuation (configural failures) and every rule's severity and verdict

In [4]:
# attenuation per cell = the weaker (smaller) of the two groups' sqrt(omega_H), where available
dim = pd.read_csv(dat_dir / 'effect_sizes' / 'dimensionality_effect_sizes.csv')
dim = dim[dim.converged == True] if 'converged' in dim.columns else dim
att = dim.groupby(['scale', 'group'])['attenuation'].min()
def cell_atten(pair, scale):
    vals = [att.get((scale, g), np.nan) for g in pair_groups[pair]]
    vals = [v for v in vals if pd.notna(v)]
    return min(vals) if vals else np.nan
es['attenuation'] = [cell_atten(p, s) for p, s in zip(es.pair, es.scale)]

# rule severities from the calibrated screen (per cell = extreme over the four transitions)
cal = pd.read_csv(dat_dir / 'cfa_calibrated_criteria' / 'calibrated_screen.csv')
cal['dmi_cfi_margin'] = cal.delta_cfi - cal.dmi_cfi_p05          # negative = flagged by DMI-CFI
cal['dmi_rmsea_margin'] = cal.delta_rmsea - cal.dmi_rmsea_p95    # positive = flagged by DMI-RMSEA
sev = cal.groupby(['pair', 'scale']).agg(
    min_delta_cfi=('delta_cfi', 'min'), max_delta_rmsea=('delta_rmsea', 'max'),
    max_rmsea_d=('rmsea_d', 'max'), min_p_lrt=('p_lrt', 'min'),
    min_dmi_cfi_margin=('dmi_cfi_margin', 'min'), max_dmi_rmsea_margin=('dmi_rmsea_margin', 'max')).reset_index()

# verdicts (first failing level per rule) and the permutation reference
ver = pd.read_csv(dat_dir / 'cfa_calibrated_criteria' / 'verdicts_by_method.csv')
perm_h = pd.read_csv(dat_dir / 'cfa' / 'orig_cfa_res.csv'); perm_o = pd.read_csv(dat_dir / 'cfa_other_gp_en' / 'orig_cfa_res.csv')
perm = pd.concat([perm_h, perm_o], ignore_index=True)
perm['pair'] = perm['pair'].map({'V_GP': 'val_gp', 'V_EN': 'val_en', 'GP_EN': 'gp_en'})
pcols = ['pconfig', 'pthresholds', 'pmetric', 'pscalar', 'pstrict']
for c in pcols: perm[c] = pd.to_numeric(perm[c], errors='coerce')
perm['perm_min_p'] = perm[pcols].min(axis=1)        # NaN configural (non-convergence) -> treat as 0 below
perm.loc[perm.pconfig.isna(), 'perm_min_p'] = 0.0
perm = perm[['pair', 'scale', 'perm_min_p']]

df = es.merge(sev, on=['pair', 'scale'], how='left').merge(ver, on=['pair', 'scale'], how='left').merge(perm, on=['pair', 'scale'], how='left')
RULES = ['conv', 'lrt', 'rmsea_d_05', 'rmsea_d_08', 'dmi05', 'dmi01']
for r in RULES: df[f'flag_{r}'] = df[r] != 'pass'
df['flag_perm'] = df['perm_fail'] != 'pass'
df['flag_perm_delta'] = df['perm_fail'].isin(['thresholds', 'metric', 'scalar', 'strict'])
df.to_csv(out_dir / 'cells_effects_and_rules.csv', index=False)
print(df[['pair', 'scale', 'perm_fail', 'gross_bias_pct', 'net_bias_pct', 'net_ci_excl0', 'max_dmacs', 'attenuation']].round(2).to_string(index=False))

  pair                   scale  perm_fail  gross_bias_pct  net_bias_pct  net_ci_excl0  max_dmacs  attenuation
val_gp    anhedonic_depression configural            2.22         -0.27         False       0.23         0.82
val_gp           anxious_worry thresholds            3.78         -0.68         False       0.26          NaN
val_gp           appetite_gain     metric            1.16         -0.13         False       0.08          NaN
val_gp           appetite_loss thresholds            2.68          0.70         False       0.17          NaN
val_gp      cognitive_problems     scalar            2.19         -0.39         False       0.17          NaN
val_gp              hyposomnia configural            2.02         -0.76         False       0.16          NaN
val_gp          indecisiveness     scalar            2.57         -0.60         False       0.15          NaN
val_gp                insomnia       pass            1.40         -0.37         False       0.09          NaN
val_gp    

## 3. Rank-order agreement: criterion severity vs consequence size

In [5]:
SEVERITY = {  # (column, direction: +1 = larger is more severe)
    'conventional: −min ΔCFI': ('min_delta_cfi', -1), 'conventional: max ΔRMSEA': ('max_delta_rmsea', +1),
    'LRT: −log10 min p': ('min_p_lrt', -1), 'RMSEA_D: max': ('max_rmsea_d', +1),
    'DMI: −min ΔCFI margin': ('min_dmi_cfi_margin', -1), 'DMI: max ΔRMSEA margin': ('max_dmi_rmsea_margin', +1),
    'permutation: −log10 min p': ('perm_min_p', -1)}
EFFECTS = {'gross bias % range': 'gross_bias_pct', '|net bias| % range': 'abs_net_bias_pct', 'max dMACS': 'max_dmacs'}
d = df[df.converged == True].copy()
for c in ['min_p_lrt', 'perm_min_p']:
    d[c] = -np.log10(d[c].clip(lower=1e-4))   # -log10 so that larger = more severe; sign handled below
rows = []
for sname, (col, sign) in SEVERITY.items():
    x = d[col] * (sign if col not in ('min_p_lrt', 'perm_min_p') else 1)
    for ename, ecol in EFFECTS.items():
        m = x.notna() & d[ecol].notna()
        rho, p = stats.spearmanr(x[m], d.loc[m, ecol])
        rows.append(dict(severity=sname, effect=ename, n=int(m.sum()), spearman_rho=round(rho, 3), p=round(p, 4)))
sp = pd.DataFrame(rows)
sp.to_csv(out_dir / 'spearman_severity_vs_effect.csv', index=False)
print(sp.pivot(index='severity', columns='effect', values='spearman_rho').to_string())

effect                     gross bias % range  max dMACS  |net bias| % range
severity                                                                    
DMI: max ΔRMSEA margin                  0.249      0.125               0.355
DMI: −min ΔCFI margin                   0.562      0.624               0.402
LRT: −log10 min p                       0.540      0.605               0.319
RMSEA_D: max                            0.595      0.534               0.383
conventional: max ΔRMSEA                0.150     -0.047               0.297
conventional: −min ΔCFI                 0.585      0.600               0.393
permutation: −log10 min p               0.210      0.274               0.259


## 4. Precision and recall of each binary rule against practical-significance definitions

Definitions (a cell is "consequential" if):
- `dmacs20` / `dmacs40`: largest item dMACS ≥ .20 / ≥ .40 (Nye & Drasgow-type benchmarks);
- `gross2.5` / `gross5`: gross |DIF bias| ≥ 2.5% / ≥ 5% of scale range;
- `net_ci`: net sum-score bias 95% CI excludes zero;
- `atten90`: criterion-correlation attenuation ≤ .90 (ωH ≤ .81) in either group (configural failures only; NaN elsewhere = not consequential);
- `any`: dmacs40 or gross5 or net_ci or atten90.

In [6]:
d = df[df.converged == True].copy()
PS = {
    'dmacs20': d.max_dmacs >= 0.20, 'dmacs40': d.max_dmacs >= 0.40,
    'gross2.5': d.gross_bias_pct >= 2.5, 'gross5': d.gross_bias_pct >= 5.0,
    'net_ci': d.net_ci_excl0.astype(bool),
    'atten90': d.attenuation.fillna(1.0) <= 0.90,
}
PS['any'] = PS['dmacs40'] | PS['gross5'] | PS['net_ci'] | PS['atten90']
FLAGS = {**{r: d[f'flag_{r}'] for r in RULES}, 'permutation': d.flag_perm}
out = []
for psname, ps in PS.items():
    for rname, fl in FLAGS.items():
        tp = int((fl & ps).sum()); fp = int((fl & ~ps).sum()); fn = int((~fl & ps).sum()); tn = int((~fl & ~ps).sum())
        prec = tp / (tp + fp) if tp + fp else np.nan; rec = tp / (tp + fn) if tp + fn else np.nan
        # Cohen's kappa
        n = tp + fp + fn + tn; po = (tp + tn) / n; pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n**2
        kappa = (po - pe) / (1 - pe) if pe < 1 else np.nan
        out.append(dict(practical_def=psname, n_consequential=int(ps.sum()), rule=rname, n_flagged=int(fl.sum()),
                        tp=tp, fp=fp, fn=fn, tn=tn, precision=round(prec, 2) if pd.notna(prec) else np.nan,
                        recall=round(rec, 2) if pd.notna(rec) else np.nan, kappa=round(kappa, 2) if pd.notna(kappa) else np.nan))
pr = pd.DataFrame(out)
pr.to_csv(out_dir / 'criteria_vs_practical.csv', index=False)
pd.set_option('display.width', 250)
for psname in PS:
    sub = pr[pr.practical_def == psname]
    print(f"\n=== consequential by '{psname}': {int(sub.n_consequential.iloc[0])} of {len(d)} cells ===")
    print(sub[['rule', 'n_flagged', 'tp', 'fp', 'fn', 'tn', 'precision', 'recall', 'kappa']].to_string(index=False))
log('DONE practical-significance comparison')


=== consequential by 'dmacs20': 19 of 48 cells ===
       rule  n_flagged  tp  fp  fn  tn  precision  recall  kappa
       conv         22   8  14  11  15       0.36    0.42  -0.06
        lrt         43  19  24   0   5       0.44    1.00   0.14
 rmsea_d_05         43  19  24   0   5       0.44    1.00   0.14
 rmsea_d_08         33  18  15   1  14       0.55    0.95   0.38
      dmi05         43  19  24   0   5       0.44    1.00   0.14
      dmi01         34  17  17   2  12       0.50    0.89   0.27
permutation         40  18  22   1   7       0.45    0.95   0.16

=== consequential by 'dmacs40': 1 of 48 cells ===
       rule  n_flagged  tp  fp  fn  tn  precision  recall  kappa
       conv         22   1  21   0  26       0.05     1.0   0.05
        lrt         43   1  42   0   5       0.02     1.0   0.00
 rmsea_d_05         43   1  42   0   5       0.02     1.0   0.00
 rmsea_d_08         33   1  32   0  15       0.03     1.0   0.02
      dmi05         43   1  42   0   5       0.02   

## Notes

- Effect sizes come from the metric model of each cell (common loadings/thresholds, free
  group-2 intercepts); for cells where configural or metric invariance already fails they
  are approximations (see `hitop_cfa.effect_size` docstring).
- `perm_min_p` is the smallest permutation p across the ladder (configural non-convergence
  coded as 0); `min_p_lrt` the smallest scaled-Δχ² p across the four transitions.